# grok-003 · 双卡 T4 + 指令 SFT（学习向）

> **你在学什么**：在 **两张 T4 都可见** 时，如何用 `DataParallel`（简单多卡）做更大 batch 的 LoRA SFT。
>
> **关键概念**：
> - 双卡 ≠ 一张 32GB，默认是 **数据并行**（每张卡一份模型副本）
> - 公开小指令集（如 Alpaca 子集）+ train/val
> - 与 002 对比：单卡锁死 vs 双卡吞吐
>
> **局限**：`DataParallel` 简单但不总是最快；工业界更常用 DDP/Accelerate（后续深入）。


# grok-003-dual-t4-instruct-lora

## 命名约定
| Notebook | 含义 |
|---|---|
| `grok-001-t4-smoke-cnn` | T4 冒烟：GEMM + 小 CNN |
| `grok-002-single-t4-qwen-lora` | **1×T4**（`CUDA_VISIBLE_DEVICES=0`）Qwen0.5B LoRA |
| **`grok-003-dual-t4-instruct-lora`** | **2×T4** 可见，公开小指令集 + LoRA SFT + 简单 val |

## 本 notebook 做什么
1. **不**锁定单卡：使用 Kaggle `NvidiaTeslaT4` 套餐的 **两张 T4**
2. 从 HuggingFace 拉取精简指令数据（优先 `tatsu-lab/alpaca` 子集；失败则用内置扩展集）
3. **Qwen2.5-0.5B-Instruct + fp16 LoRA**，`DataParallel` 吃双卡 batch
4. train/val 划分、loss 曲线、固定探针 before/after、保存 adapter
5. 写出 `grok003_results.json`（含 1卡 vs 本 run 的设定说明，便于和 002 对比）

> 仍控制在免费 T4 配额内：约 1–2k 样本、短 step，重点是 **双卡流水线 + 更像真项目的数据**。


In [ ]:
# 【步骤】锁定单卡：只让进程看见 GPU0（学习「逻辑单卡」）
# -*- dual-T4: do NOT set CUDA_VISIBLE_DEVICES=0 -*-
import os
# Explicitly clear any pin so both T4s are visible
os.environ.pop('CUDA_VISIBLE_DEVICES', None)
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')
print('CUDA_VISIBLE_DEVICES unset -> use all visible GPUs')


In [ ]:
# 【步骤】检查有几张 GPU、名字是否为 Tesla T4
# -*- setup -*-
import os, json, time, math, random, platform, traceback, subprocess, sys
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

OUT = Path('/kaggle/working')
OUT.mkdir(parents=True, exist_ok=True)

print('python', platform.python_version())
print('torch', torch.__version__)
assert torch.cuda.is_available(), 'GPU required'
n = torch.cuda.device_count()
print('device_count', n)
for i in range(n):
    p = torch.cuda.get_device_properties(i)
    print(f'  [{i}] {torch.cuda.get_device_name(i)}  {p.total_memory/1e9:.1f}GB  cap={torch.cuda.get_device_capability(i)}')
assert n >= 2, f'grok-003 expects 2 visible T4s, got {n}. Check machineShape=NvidiaTeslaT4 and no CUDA_VISIBLE_DEVICES pin.'
assert all('T4' in torch.cuda.get_device_name(i) for i in range(n)), 'expected Tesla T4 GPUs'

DEVICE = torch.device('cuda:0')  # 主设备：默认第一张可见 GPU
AMP_DTYPE = torch.float16  # T4 SM75: fp16 not bf16  # T4 无 fp16（无原生 bf16 tensor core）
print('primary', DEVICE, 'amp', AMP_DTYPE)


In [ ]:
# 【步骤】可选：拉取公开指令数据切片（失败则走内置数据）
# -*- dataset: HF alpaca subset with robust fallback -*-
N_TRAIN_TARGET = 1500
N_VAL = 150
MAX_LEN = 384

BUILTIN = [
    ('What is 17 * 19?', '323'),
    ('Explain LoRA in one sentence.', 'LoRA freezes the base model and trains small low-rank adapters in linear layers.'),
    ('Python function add(a,b) returning the sum.', 'def add(a, b):\n    return a + b'),
    ('What does PEFT stand for?', 'Parameter-Efficient Fine-Tuning'),
    ('GCD of 48 and 18?', '6'),
    ('Translate to Chinese: Machine learning is powerful.', '机器学习很强大。'),
    ('SQL: all rows from users where active=1', 'SELECT * FROM users WHERE active = 1;'),
    ('Is 91 prime? If not factor it.', 'No. 91 = 7 * 13.'),
    ('Why use cosine learning rate decay?', 'It smoothly anneals LR and often improves late-stage convergence.'),
    ('JSON object for Alice age 30', '{"name": "Alice", "age": 30}'),
]

def expand_builtin(n=1600):
    base = []
    templates = [
        lambda q,a: (q,a),
        lambda q,a: (f'### Instruction:\n{q}\n### Response:', a),
        lambda q,a: (f'Q: {q}\nA:', a),
        lambda q,a: (f'Please answer briefly: {q}', a),
    ]
    i = 0
    while len(base) < n:
        q,a = BUILTIN[i % len(BUILTIN)]
        # slight paraphrase noise for variety
        q2 = q if (i % 5) else f'{q} (be concise)'
        base.append(templates[i % len(templates)](q2, a))
        i += 1
    return base

pairs = []
data_source = 'builtin_expanded'
try:
    from datasets import load_dataset
    print('loading tatsu-lab/alpaca (stream/subset)...')
    # small slice only
    ds = load_dataset('tatsu-lab/alpaca', split=f'train[:{N_TRAIN_TARGET + N_VAL + 50}]')
    for row in ds:
        instr = (row.get('instruction') or '').strip()
        inp = (row.get('input') or '').strip()
        out = (row.get('output') or '').strip()
        if not instr or not out:
            continue
        if inp:
            instr = f'{instr}\n{inp}'
        pairs.append((instr, out))
    data_source = f'tatsu-lab/alpaca first {len(pairs)} rows'
    print('alpaca pairs', len(pairs))
except Exception as e:
    print('HF alpaca failed, using builtin expansion:', repr(e)[:300])
    pairs = expand_builtin(N_TRAIN_TARGET + N_VAL + 50)
    data_source = 'builtin_expanded'

random.Random(SEED).shuffle(pairs)
pairs = pairs[: N_TRAIN_TARGET + N_VAL]
val_pairs = pairs[:N_VAL]
train_pairs = pairs[N_VAL:]
print('data_source', data_source)
print('train', len(train_pairs), 'val', len(val_pairs))
print('train sample:', train_pairs[0][0][:120], '->', train_pairs[0][1][:80])


In [ ]:
# 【步骤】检查有几张 GPU、名字是否为 Tesla T4
# -*- model: Qwen2.5-0.5B-Instruct + LoRA, then DataParallel -*-
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, TaskType
print('transformers', transformers.__version__)

# peft torchao harden (same class of bug as grok-002)
try:
    import peft.tuners.lora.torchao as peft_torchao
    _orig = peft_torchao.is_torchao_available
    def _safe():
        try:
            return bool(_orig())
        except Exception:
            return False
    peft_torchao.is_torchao_available = _safe
    print('patched peft torchao check')
except Exception as e:
    print('torchao patch skipped', e)

MODEL_ID = 'Qwen/Qwen2.5-0.5B-Instruct'
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.float16,
    trust_remote_code=True,
)
base.config.use_cache = False
base.gradient_checkpointing_enable()  # 用算力换显存：激活重计算
if hasattr(base, 'enable_input_require_grads'):  # 梯度检查点时常需要，否则 LoRA 可能收不到梯度
    base.enable_input_require_grads()

cand = ['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj']
present = sorted({n.split('.')[-1] for n,_ in base.named_modules() if n.split('.')[-1] in cand})
print('lora targets', present)
assert present

lora_cfg = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    target_modules=present,
)
model = get_peft_model(base, lora_cfg)
model.print_trainable_parameters()
model.to(DEVICE)

# DataParallel for dual T4 (simple, good enough for this size)
if torch.cuda.device_count() > 1:
    model = nn.DataParallel(model)
    print('wrapped DataParallel on', torch.cuda.device_count(), 'GPUs')

def raw_model(m):
    return m.module if isinstance(m, nn.DataParallel) else m


In [ ]:
# 【步骤】简单多卡：DataParallel（易用；真·高效多用 DDP/Accelerate）
# -*- collate / loaders -*-
def format_chat(instr, resp):
    messages = [
        {'role':'user','content':instr},
        {'role':'assistant','content':resp},
    ]
    if hasattr(tokenizer, 'apply_chat_template'):
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return f'<|user|>\n{instr}\n<|assistant|>\n{resp}{tokenizer.eos_token}'

class InstrDS(Dataset):
    def __init__(self, pairs, max_len=MAX_LEN):
        self.rows = []
        for instr, resp in pairs:
            text = format_chat(instr, resp)
            self.rows.append(tokenizer(text, truncation=True, max_length=max_len, padding=False))
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, i):
        return self.rows[i]

def collate(batch):
    return tokenizer.pad(batch, padding=True, return_tensors='pt')

# Dual T4: larger per-step batch (2 per GPU * 2 GPUs via DP gathers)
BATCH = 4  # global batch into DataParallel; each GPU sees BATCH/num_gpus roughly
train_loader = DataLoader(InstrDS(train_pairs), batch_size=BATCH, shuffle=True, collate_fn=collate, drop_last=True)
val_loader = DataLoader(InstrDS(val_pairs), batch_size=BATCH, shuffle=False, collate_fn=collate)
print('train batches', len(train_loader), 'val batches', len(val_loader), 'batch', BATCH)


In [ ]:
# -*- generate + BEFORE -*-
PROBES = [
    'What is 17 * 19?',
    'Explain LoRA in one sentence.',
    'Write a Python function add(a, b) that returns the sum.',
    'What does PEFT stand for?',
]

@torch.no_grad()
def generate(prompt, max_new=80):
    m = raw_model(model)
    m.eval()
    messages = [{'role':'user','content':prompt}]
    if hasattr(tokenizer, 'apply_chat_template'):
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    else:
        text = f'<|user|>\n{prompt}\n<|assistant|>\n'
    inputs = tokenizer(text, return_tensors='pt').to(DEVICE)
    out = m.generate(
        **inputs,
        max_new_tokens=max_new,
        do_sample=False,  # greedy：评测可复现，避免采样噪声
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    gen = out[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(gen, skip_special_tokens=True).strip()

before = {p: generate(p) for p in PROBES}
print('=== BEFORE ===')
for p,g in before.items():
    print('Q:', p); print('A:', g[:350]); print('---')


In [ ]:
# 【步骤】简单多卡：DataParallel（易用；真·高效多用 DDP/Accelerate）
# -*- train on 2xT4 -*-
m_raw = raw_model(model)
trainable = [p for p in m_raw.parameters() if p.requires_grad]
n_train = sum(p.numel() for p in trainable)
n_all = sum(p.numel() for p in m_raw.parameters())
print(f'trainable {n_train:,} / {n_all:,} ({100*n_train/max(1,n_all):.3f}%)')

opt = torch.optim.AdamW(trainable, lr=2e-4, weight_decay=0.0)
STEPS = 200
WARMUP = 15

def lr_at(step):
    if step < WARMUP:
        return (step+1)/WARMUP
    t = (step - WARMUP)/max(1, STEPS-WARMUP)
    return 0.5*(1+math.cos(math.pi*t))

scaler = torch.amp.GradScaler('cuda', enabled=True)
torch.cuda.reset_peak_memory_stats()
model.train()

def run_loss(batch):
    batch = {k:v.to(DEVICE) for k,v in batch.items()}
    labels = batch['input_ids'].clone()
    labels[batch['attention_mask']==0] = -100  # -100：CrossEntropy 忽略 pad（及可选 prompt）位置
    with torch.amp.autocast('cuda', dtype=AMP_DTYPE):
        # DataParallel forward: call module with kwargs via underlying if needed
        out = model(input_ids=batch['input_ids'], attention_mask=batch['attention_mask'], labels=labels)
        # DP may return mean of losses depending on version; HF CausalLMOutputWithPast has .loss
        loss = out.loss if not isinstance(out, (tuple, list)) else out[0]
        if hasattr(loss, 'mean'):
            loss = loss.mean()
    return loss

losses = []
t0 = time.perf_counter()
it = iter(train_loader)
for step in range(STEPS):
    try:
        batch = next(it)
    except StopIteration:
        it = iter(train_loader)
        batch = next(it)
    for g in opt.param_groups:
        g['lr'] = 2e-4 * lr_at(step)
    opt.zero_grad(set_to_none=True)
    loss = run_loss(batch)
    scaler.scale(loss).backward()
    scaler.unscale_(opt)
    torch.nn.utils.clip_grad_norm_(trainable, 1.0)  # 梯度裁剪：防爆炸
    scaler.step(opt)
    scaler.update()
    losses.append(float(loss.detach().float().cpu()))
    if step % 20 == 0 or step == STEPS-1:
        print(f'step {step:03d}/{STEPS} loss={losses[-1]:.4f} lr={opt.param_groups[0]["lr"]:.2e}')

train_s = time.perf_counter() - t0
peak_gb = torch.cuda.max_memory_allocated() / 1e9
print(f'train_s={train_s:.1f} peak_mem_gb={peak_gb:.2f}')

# quick val
model.eval()
val_losses = []
with torch.no_grad():
    for batch in val_loader:
        val_losses.append(float(run_loss(batch).detach().float().cpu()))
val_loss = sum(val_losses)/max(1,len(val_losses))
print('val_loss', val_loss)


In [ ]:
# 【步骤】检查有几张 GPU、名字是否为 Tesla T4
# -*- AFTER + save -*-
after = {p: generate(p) for p in PROBES}
print('=== AFTER ===')
for p,g in after.items():
    print('Q:', p); print('A:', g[:350]); print('---')

adapter_dir = OUT / 'grok003_adapter'
adapter_dir.mkdir(exist_ok=True)
raw_model(model).save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)

results = {
    'notebook': 'grok-003-dual-t4-instruct-lora',
    'phase': 'dual_t4',
    'naming_series': {
        '001': 'grok-001-t4-smoke-cnn',
        '002': 'grok-002-single-t4-qwen-lora',
        '003': 'grok-003-dual-t4-instruct-lora',
    },
    'machine_shape_requested': 'NvidiaTeslaT4',
    'cuda_visible_devices': os.environ.get('CUDA_VISIBLE_DEVICES'),
    'device_count': torch.cuda.device_count(),
    'device_names': [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
    'parallelism': 'DataParallel',
    'model_id': MODEL_ID,
    'mode': 'lora_fp16',
    'data_source': data_source,
    'n_train': len(train_pairs),
    'n_val': len(val_pairs),
    'batch_size': BATCH,
    'steps': STEPS,
    'loss_start': losses[0],
    'loss_end': losses[-1],
    'loss_min': min(losses),
    'val_loss': val_loss,
    'trainable_params': n_train,
    'total_params': n_all,
    'trainable_pct': 100.0 * n_train / max(1,n_all),
    'train_seconds': train_s,
    'peak_mem_gb': peak_gb,
    'before': before,
    'after': after,
    'artifact': str(adapter_dir),
    'vs_002': '002 forced 1xT4 + tiny handset; 003 uses 2xT4 + larger public/expanded instruct set + val split + DP',
}
path = OUT / 'grok003_results.json'
path.write_text(json.dumps(results, indent=2, ensure_ascii=False))
print('wrote', path)
print(json.dumps({k: results[k] for k in [
    'device_count','device_names','data_source','n_train','n_val',
    'loss_start','loss_end','val_loss','trainable_pct','train_seconds','peak_mem_gb'
]}, indent=2, ensure_ascii=False))
print('DONE grok-003-dual-t4-instruct-lora')


## 学习检查清单

- 你应能回答：DataParallel 在做什么？双卡为何不是 32GB 显存池？
- 建议：改一个超参重跑一小段，观察 log 变化（比只读代码更有效）。
